In [ ]:
# poker_env.py
import random
from itertools import combinations, groupby
from collections import Counter

class PokerEnv:
    """
    Entorno simplificado de póker estilo 'Texas-like' (pero con 2 cartas por jugador
    y 4 cartas en mesa; se usan 5 de las 6 cartas para formar la mejor mano).
    
    Cartas: (valor, suit)
      - valor: 2..14 (14 = A)
      - suit: 0..3 (picas, corazones, diamantes, tréboles) -- color no es necesario
    Acciones:
      0 -> fold (retirarse)
      1 -> continue (jugar hasta showdown o siguiente ronda)
    """

    def __init__(self, reveal_rounds=4):
        # reveal_rounds: cuántas cartas de mesa se van a revelar en total (aquí 4)
        self.reveal_rounds = reveal_rounds
        self.reset()

    def reset(self, seed=None):
        if seed is not None:
            random.seed(seed)
        # crear mazo
        valores = list(range(2, 15))  # 2..14 (A=14)
        suits = [0,1,2,3]
        self.deck = [(v,s) for v in valores for s in suits]
        random.shuffle(self.deck)

        # repartir: 2 cartas por cada uno
        self.players_hands = [ [self.deck.pop(), self.deck.pop()] for _ in range(2) ]  # jugador0, jugador1
        self.table_cards = []  # cartas de la mesa (se irán añadiendo hasta reveal_rounds)
        self.round = 0
        # track de fold por jugador: True si se retiró
        self.folded = [False, False]
        return self.get_observation()

    def deal_table_card(self, n=1):
        for _ in range(n):
            if self.deck:
                self.table_cards.append(self.deck.pop())

    def get_observation(self, player_index=None):
        """
        Observación simple: cartas propias + cartas visibles en mesa + round
        (player_index opcional para retornar sólo la mano del jugador)
        """
        obs = {
            "round": self.round,
            "table": list(self.table_cards),
            "hands": [list(self.players_hands[0]), list(self.players_hands[1])]
        }
        if player_index is not None:
            obs["my_hand"] = list(self.players_hands[player_index])
        return obs

    # ------------------------
    # Evaluación de manos
    # ------------------------
    @staticmethod
    def _is_flush(cards):
        suits = [s for _,s in cards]
        return len(set(suits)) == 1

    @staticmethod
    def _is_straight(values):
        """Asume valores ordenados únicos. Devuelve (es_straight, high_value)."""
        # manejar Wheel (A-2-3-4-5) si A=14
        vals = sorted(set(values))
        if len(vals) < 5:
            return False, None
        # verifica secuencias de longitud 5
        for i in range(len(vals) - 4):
            window = vals[i:i+5]
            if window[0]+4 == window[-1] and all(window[j]+1 == window[j+1] for j in range(4)):
                return True, window[-1]
        # special wheel: A(14),2,3,4,5 -> treat A as 1
        if set([14,2,3,4,5]).issubset(set(values)):
            return True, 5
        return False, None

    @staticmethod
    def _group_by_value(cards):
        vals = [v for v,_ in cards]
        counter = Counter(vals)
        # Return list of groups by count descending then value desc
        groups = sorted(counter.items(), key=lambda x: (x[1], x[0]), reverse=True)
        return groups  # [(value, count), ...]

    @classmethod
    def evaluate_5cards(cls, cards5):
        """
        Evalúa una lista de 5 cartas.
        Retorna una tupla (rank, tiebreaker_tuple) donde rank más alto = mejor mano.
        Rankings: (10 royal flush, 9 straight flush, 8 four, 7 full, 6 flush, 5 straight,
                   4 three, 3 two pair, 2 one pair, 1 high card)
        tiebreaker: tupla de valores para desempate en orden descendente.
        """
        values = [v for v,_ in cards5]
        values_sorted = sorted(values, reverse=True)
        is_flush = cls._is_flush(cards5)
        is_straight, high_straight = cls._is_straight(values)
        groups = cls._group_by_value(cards5)  # sorted by count desc then value desc

        # Royal / Straight flush
        if is_flush and is_straight:
            if set(values) == set([10,11,12,13,14]):
                return (10, (14,))  # royal flush
            return (9, (high_straight,))  # straight flush

        # Four of a kind
        if groups[0][1] == 4:
            four_val = groups[0][0]
            kicker = max(v for v in values if v != four_val)
            return (8, (four_val, kicker))

        # Full house (3 + 2)
        if groups[0][1] == 3 and groups[1][1] == 2:
            return (7, (groups[0][0], groups[1][0]))

        # Flush
        if is_flush:
            return (6, tuple(values_sorted))

        # Straight
        if is_straight:
            return (5, (high_straight,))

        # Three of a kind
        if groups[0][1] == 3:
            trips = groups[0][0]
            kickers = sorted([v for v in values if v != trips], reverse=True)
            return (4, (trips, ) + tuple(kickers))

        # Two pair
        if groups[0][1] == 2 and groups[1][1] == 2:
            high_pair = groups[0][0]
            low_pair = groups[1][0]
            kicker = max(v for v in values if v != high_pair and v != low_pair)
            return (3, (high_pair, low_pair, kicker))

        # One pair
        if groups[0][1] == 2:
            pair = groups[0][0]
            kickers = sorted([v for v in values if v != pair], reverse=True)
            return (2, (pair,) + tuple(kickers))

        # High card
        return (1, tuple(values_sorted))

    # ------------------------
    # Mejor mano entre 6 cartas -> tomar combinaciones de 5
    # ------------------------
    @classmethod
    def best_hand_from_six(cls, six_cards):
        best_rank = (-1, ())
        best_comb = None
        for comb in combinations(six_cards, 5):
            rank = cls.evaluate_5cards(list(comb))
            if rank > best_rank:
                best_rank = rank
                best_comb = comb
        return best_rank, best_comb

    # ------------------------
    # Step: ejecutar una acción de un jugador
    # ------------------------
    def step(self, player_index, action):
        """
        player_index: 0 o 1
        action: 0 fold, 1 continue
        Devuelve: obs, reward_for_player, done, info
        NOTE: Esta función no gestiona turnos complejos ni apuestas; es minimal.
        """
        if self.folded[player_index]:
            return self.get_observation(player_index), 0.0, True, {"msg":"player already folded"}

        if action == 0:
            # fold -> jugador pierde automáticamente (reward negativo local)
            self.folded[player_index] = True
            # Si el otro ya está folded -> terminar
        elif action == 1:
            # continuar: si aún no se revelaron todas las cartas, revelar la siguiente
            if self.round < self.reveal_rounds:
                self.deal_table_card(1)
            # sino, ya estamos en showdown
        else:
            raise ValueError("acción inválida")

        # avanzar round si se reveló carta
        # ya controlado por calling code si lo desea

        # check terminal:
        done = False
        if all(self.folded) or self.round >= self.reveal_rounds and (not any(not f for f in self.folded)):
            done = True

        # En este entorno manejaremos el resultado en la función externa (Game)
        return self.get_observation(player_index), 0.0, False, {}

# ------------------------
# Motor de juego (coordinador)
# ------------------------
class Game:
    """
    Gestiona partidas entre dos agentes que implementen:
      - reset() opcional
      - move(obs, player_index) -> acción (0/1)
      - update(obs, player_index) opcional
      - reward(reward, player_index) opcional
    """

    def __init__(self, player1, player2, env=None):
        self.players = [player1, player2]
        self.env = env if env is not None else PokerEnv()
    
    def selfplay(self, rounds=1000, verbose=False, allow_fold=True):
        wins = [0,0]
        ties = 0
        for ep in range(rounds):
            obs = self.env.reset()
            # opcional reset de agentes si lo requieren
            for p in self.players:
                if hasattr(p, "reset"):
                    p.reset()

            # decisión simple por rondas; cada jugador decide en su turno.
            # aquí simplificamos: en cada "ronda" ambos jugadores pueden decidir fold/play
            done = False
            # revelar 0 cartas al inicio, luego hasta reveal_rounds
            # for simplicidad, permitimos max reveal_rounds iteraciones
            for r in range(self.env.reveal_rounds + 1):
                self.env.round = r
                # cada jugador actúa (si no folded)
                for pid, player in enumerate(self.players):
                    if self.env.folded[pid]:
                        continue
                    obs_player = self.env.get_observation(pid)
                    action = player.move(obs_player, pid) if hasattr(player, "move") else random.choice([0,1])
                    # si no permitimos fold en fase de entrenamiento, override:
                    if not allow_fold:
                        action = 1
                    # aplicar acción: si action==1 y quedan cartas por revelar, se revela
                    if action == 1 and r < self.env.reveal_rounds:
                        self.env.deal_table_card(1)
                    elif action == 0:
                        self.env.folded[pid] = True
                    # opcional update al agente
                    if hasattr(player, "update"):
                        player.update(self.env.get_observation(pid))
                # check terminal: si uno se retiró y el otro no, termina antes
                if self.env.folded.count(True) == 1:
                    break
            # showdown / determinar ganador
            # si ambos se retiraron (caso raro), empatar
            if self.env.folded[0] and self.env.folded[1]:
                ties += 1
                # notificar recompensas
                for pid, p in enumerate(self.players):
                    if hasattr(p, "reward"):
                        p.reward(0.5)
                continue
            # si uno fold -> el otro gana
            if self.env.folded[0] and not self.env.folded[1]:
                wins[1] += 1
                if hasattr(self.players[1], "reward"):
                    self.players[1].reward(1)
                if hasattr(self.players[0], "reward"):
                    self.players[0].reward(0)
                continue
            if self.env.folded[1] and not self.env.folded[0]:
                wins[0] += 1
                if hasattr(self.players[0], "reward"):
                    self.players[0].reward(1)
                if hasattr(self.players[1], "reward"):
                    self.players[1].reward(0)
                continue

            # Ambos jugaron hasta el showdown => comparar mejores manos
            six0 = list(self.env.players_hands[0]) + list(self.env.table_cards)
            six1 = list(self.env.players_hands[1]) + list(self.env.table_cards)
            rank0, comb0 = self.env.best_hand_from_six(six0)
            rank1, comb1 = self.env.best_hand_from_six(six1)

            if rank0 > rank1:
                wins[0] += 1
                if hasattr(self.players[0], "reward"):
                    self.players[0].reward(1)
                if hasattr(self.players[1], "reward"):
                    self.players[1].reward(0)
            elif rank1 > rank0:
                wins[1] += 1
                if hasattr(self.players[1], "reward"):
                    self.players[1].reward(1)
                if hasattr(self.players[0], "reward"):
                    self.players[0].reward(0)
            else:
                # empate (rank igual) -> empate
                ties += 1
                if hasattr(self.players[0], "reward"):
                    self.players[0].reward(0.5)
                if hasattr(self.players[1], "reward"):
                    self.players[1].reward(0.5)

        return {"wins": wins, "ties": ties}

# ------------------------
# Agente de ejemplo: aleatorio
# ------------------------
class RandomAgent:
    def __init__(self, prob_play=0.8):
        self.prob_play = prob_play
    def reset(self):
        pass
    def move(self, obs, player_index=None):
        # simple: decide si jugar o retirarse
        return 1 if random.random() < self.prob_play else 0
    def update(self, obs):
        pass
    def reward(self, r):
        pass

# ------------------------
# Ejemplo de uso rápido
# ------------------------
if __name__ == "__main__":
    a1 = RandomAgent(prob_play=0.9)
    a2 = RandomAgent(prob_play=0.9)
    game = Game(a1, a2, env=PokerEnv(reveal_rounds=4))
    resultado = game.selfplay(rounds=1000, allow_fold=True)
    print("Resultado:", resultado)
